# GCP head-VM loss
Reuse the five existing VMs. Stop the actual head VM, promote w3, and restart both training methods from an off-head checkpoint. Worker continuation and GCS-state restoration are not tested. The original notebook is unchanged.


In [ ]:
import importlib
import json
import sys
from pathlib import Path

# Start Jupyter in gossip_benchmarks, or adjust this path.
HELPERS = Path("head_vm_loss").resolve()
sys.path.insert(0, str(HELPERS))
launcher = importlib.import_module("launcher")

importlib.reload(launcher)

STATE_FILE = "ray_gcp_results/rayexp-1013-194805-c4ad9f-state.json"
WORKDIR = "ray_gcp_results/head-vm-loss"
experiment = launcher.HeadVMLoss(STATE_FILE, workdir=WORKDIR)


In [ ]:
# Read-only: includes older labeled clusters. Creates/deletes nothing.
print(json.dumps(experiment.inventory(), indent=2))


w1/w2 train; w4 hosts NFS, driver and evidence; w3 becomes the new head. The notebook host must remain running. Plain helper source is in `head_vm_loss/`; details are in its short README.


In [ ]:
RUN_HEAD_LOSS = False
SUITE = "head-vm-loss-smoke-1"
OPTIONS = dict(
    repeats=1,
    steps=60,
    batch_size=32,
    model_width=512,
    checkpoint_every=30,
    fault_step=49,  # Both ranks have committed 49 updates before VM stop.
    timeout_s=1200,
    modes=("ordinary", "aligned"),  # Or ("indexed", "aligned") for the input comparison.
)
# Training uses one Torch thread and absolute 1e-5 validation tolerance.
# Existing suite names are never re-executed. Use the collection cell below.
if RUN_HEAD_LOSS:
    experiment.prepare()  # Stops idle worker-harness services; moves NFS off head.
    report = experiment.run_suite(SUITE, **OPTIONS)
    print("Suite:", report["status"])


In [ ]:
report_file = experiment.local / SUITE / "report.json"
if report_file.exists():
    report = json.loads(report_file.read_text())
    print("Status:", report["status"])
    for sample in report["samples"]:
        print(sample["mode"], "head loss" if sample["fault"] else "healthy",
              sample["status"], sample.get("validation", {}))
        if "recovery_to_new_progress_upper_bound_s" in sample:
            print("  Stop request to observed new progress (upper bound):",
                  round(sample["recovery_to_new_progress_upper_bound_s"], 2), "seconds")
    print("Evidence:", report_file.parent)


In [ ]:
# Collection works after reconnecting and does not start training.
DOWNLOAD_ONLY = False
if DOWNLOAD_ONLY:
    print(experiment.download(SUITE))


The final fault leaves the original head stopped; surviving VMs remain running. To return to the worker-loss notebook, explicitly restore its layout below. No VM deletion is included here.


In [ ]:
RESTORE_WORKER_LAYOUT = False
if RESTORE_WORKER_LAYOUT:
    experiment.restore_worker_layout()
